# Chemical Space Analysis with t-SNE

This notebook generates t-SNE (t-Distributed Stochastic Neighbor Embedding) visualizations for chemical space analysis of DeNovo HsDHODH datasets.

## Notebook Structure:

1. **Imports** - Required libraries (sklearn, rdkit, bokeh)
2. **Data Loading** - Reading druglike and highdiv datasets
3. **Fingerprint Functions** - MACCS and ECFP4
4. **create_tsne_data Function** - Calculates t-SNE for a dataset
5. **Setup** - Output directory, descriptors, and dataset names
6. **Helper Function** - Converts molecules to base64 images for tooltips
7. **Combined Visualizations** - Concatenates all datasets, removes duplicates by InChI, and generates visualizations with legend by source dataset
8. **Comparison** - Single t-SNE embedding of DRUGLIKE, HIGHDIV, and the known inhibitors, colored by source database (same color code as the physicochemical notebook)

## Processed Datasets:
- **Diversity**: DRUGLIKE, HIGHDIV
- **Descriptors**: MACCS, ECFP4
- **Datasets**: CONCAT, CRAFT, LANA, MAY

**Total**: 2 combined visualizations (HTML) + 1 comparison figure (PNG)

In [12]:
from sklearn.manifold import TSNE
import numpy as np
import pandas as pd
import rdkit
from rdkit import Chem
from rdkit import DataStructs
from rdkit.Chem import AllChem
from rdkit.Chem import MACCSkeys
from rdkit.Chem import Draw
from bokeh.plotting import figure, save, output_file
from bokeh.models import HoverTool, ColumnDataSource
import base64
from io import BytesIO

In [13]:
df_druglike = pd.read_csv('/Users/francisco/Documents/Scripts/DeNovo_HsDHODH/Analysis/Datasets/concat_datasets_druglike.csv')
df_highdiv = pd.read_csv('/Users/francisco/Documents/Scripts/DeNovo_HsDHODH/Analysis/Datasets/concat_datasets_highdiv.csv')

In [14]:
def smiles_to_maccs(smiles):
    """Converts a SMILES into MACCS fingerprint"""
    mol = Chem.MolFromSmiles(smiles)
    return MACCSkeys.GenMACCSKeys(mol) if mol else None

def smiles_to_ECFP4(smiles):
    """Converts a SMILES into ECFP4 fingerprint (Morgan with radius 2)"""
    mol = Chem.MolFromSmiles(smiles)
    return AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=1024) if mol else None

def fps_to_array(fps):
    """Fast conversion of RDKit bit vectors to a 2D numpy array.

    Uses RDKit's ConvertToNumpyArray (C implementation) instead of the
    per-bit Python conversion list(fp), which is orders of magnitude slower.
    """
    fps = list(fps)
    arr = np.zeros((len(fps), fps[0].GetNumBits()), dtype=np.int8)
    for i, fp in enumerate(fps):
        DataStructs.ConvertToNumpyArray(fp, arr[i])
    return arr

In [15]:
def create_tsne_data(df, fingerprint_type='maccs', perplexity=30, max_iter=1000):
    """
    Creates data needed to generate a t-SNE plot
    
    Args:
        df: DataFrame with 'SMILES' column
        fingerprint_type: 'maccs' or 'ecfp4'
        perplexity: The perplexity is related to the number of nearest neighbors that is used in other manifold learning algorithms.
        max_iter: Maximum number of iterations for the optimization. Should be at least 250.
    
    Returns:
        layout: t-SNE embedded coordinates (n_samples, 2)
        df: Updated DataFrame with fingerprints
    """
    # Generate fingerprints
    if fingerprint_type == 'maccs':
        df['fingerprints'] = [smiles_to_maccs(smiles) for smiles in df['SMILES']]
    elif fingerprint_type == 'ecfp4':
        df['fingerprints'] = [smiles_to_ECFP4(smiles) for smiles in df['SMILES']]
    else:
        raise ValueError("fingerprint_type must be 'maccs' or 'ecfp4'")
    
    # Remove rows without fingerprints
    df = df.dropna(subset=['fingerprints'])
    
    # Convert fingerprints to numpy array for sklearn (vectorized RDKit conversion)
    X = fps_to_array(df['fingerprints'])
    
    # Calculate t-SNE
    # Using 'jaccard' metric would be ideal for binary fingerprints, but it can be slow (O(N^2)).
    # For speed 'euclidean' is often used as a proxy, or precomputed distance matrix.
    # Given dataset sizes are small (<few k), we can try using a metric suitable for binary data if needed,
    # but default initialization usually works well.
    tsne = TSNE(n_components=2, verbose=1, perplexity=perplexity, max_iter=max_iter, random_state=42, init='pca', learning_rate='auto')
    layout = tsne.fit_transform(X)
    
    return layout, df

In [16]:
# Create results directory if it doesn't exist
import os
output_dir = '/Users/francisco/Documents/Scripts/DeNovo_HsDHODH/Analysis/Results/Chemical_Spaces/TSNE'
os.makedirs(output_dir, exist_ok=True)

# Descriptors and datasets used in the combined visualizations
fingerprint_types = ['MACCS', 'ECFP4']
dataset_names = ['CONCAT', 'CRAFT', 'LANAPDB', 'MAYBRIDGE']

In [17]:
# Function to convert molecule to base64 image
def mol_to_base64(smiles, size=(150, 150)):
    """Converts SMILES to base64 encoded PNG image"""
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return ""
        img = Draw.MolToImage(mol, size=size)
        buffered = BytesIO()
        img.save(buffered, format="PNG")
        img_str = base64.b64encode(buffered.getvalue()).decode()
        return f"data:image/png;base64,{img_str}"
    except:
        return ""

## Combined Visualization - All Datasets

In this section, we will create visualizations that combine all datasets (CONCAT, CRAFT, LANA, MAY) for each diversity type (DRUGLIKE and HIGHDIV), removing duplicates by InChI and maintaining the source dataset information.

In [18]:
# Prepare combined data removing duplicates by InChI
def prepare_combined_data(df_full):
    """
    Concatenates all datasets and removes duplicates by InChI,
    keeping source dataset information
    """
    df_combined = df_full.copy()
    
    # Remove molecules without InChI
    df_combined = df_combined.dropna(subset=['InChI'])
    print(f"  Molecules with valid InChI: {len(df_combined)}")
    
    # Remove duplicates by InChI, keeping the first occurrence
    initial_count = len(df_combined)
    df_combined = df_combined.drop_duplicates(subset=['InChI'], keep='first')
    duplicates_removed = initial_count - len(df_combined)
    print(f"  Duplicates removed: {duplicates_removed}")
    print(f"  Unique molecules: {len(df_combined)}")
    
    return df_combined

print("="*60)
print("PREPARING COMBINED DATA")
print("="*60)

# Prepare DRUGLIKE combined
print("\nDRUGLIKE:")
df_druglike_combined = prepare_combined_data(df_druglike)

# Prepare HIGHDIV combined
print("\nHIGHDIV:")
df_highdiv_combined = prepare_combined_data(df_highdiv)

print(f"\n{'='*60}")
print("Preparation complete!")
print('='*60)

PREPARING COMBINED DATA

DRUGLIKE:
  Molecules with valid InChI: 2078
  Duplicates removed: 0
  Unique molecules: 2078

HIGHDIV:
  Molecules with valid InChI: 2328
  Duplicates removed: 0
  Unique molecules: 2328

Preparation complete!


In [19]:
# Calculate t-SNEs for combined datasets
print("="*60)
print("CALCULATING COMBINED t-SNEs")
print("="*60)

tsne_combined_results = {}

combined_datasets = [
    ('DRUGLIKE', df_druglike_combined),
    ('HIGHDIV', df_highdiv_combined)
]

for diversity_type, df_combined in combined_datasets:
    print(f"\n{'='*60}")
    print(f"Processing: {diversity_type}_COMBINED ({len(df_combined)} molecules)")
    print('='*60)
    
    for descriptor in fingerprint_types:
        print(f"  Calculating t-SNE with {descriptor}...", end=' ')
        
        try:
            # Create t-SNE with appropriate perplexity
            n_samples = len(df_combined)
            perp = min(50, max(5, n_samples // 20))
            layout, df_processed = create_tsne_data(df_combined, fingerprint_type=descriptor.lower(), perplexity=perp)
            
            # Store results
            key = f"{diversity_type}_COMBINED_{descriptor}"
            tsne_combined_results[key] = {
                'layout': layout,
                'df': df_processed,
                'diversity': diversity_type,
                'descriptor': descriptor,
                'dataset': 'COMBINED',
                'perplexity': perp
            }
            
            print(f"✓ Complete ({len(df_processed)} molecules)")
            
        except Exception as e:
            print(f"✗ Error: {str(e)}")

print(f"\n{'='*60}")
print(f"Calculation complete! {len(tsne_combined_results)} combined t-SNEs calculated.")
print('='*60)

CALCULATING COMBINED t-SNEs

Processing: DRUGLIKE_COMBINED (2078 molecules)
  Calculating t-SNE with MACCS... [t-SNE] Computing 151 nearest neighbors...
[t-SNE] Indexed 2078 samples in 0.000s...
[t-SNE] Computed neighbors for 2078 samples in 0.034s...
[t-SNE] Computed conditional probabilities for sample 1000 / 2078
[t-SNE] Computed conditional probabilities for sample 2000 / 2078
[t-SNE] Computed conditional probabilities for sample 2078 / 2078
[t-SNE] Mean sigma: 1.805345
[t-SNE] KL divergence after 250 iterations with early exaggeration: 71.451294
[t-SNE] KL divergence after 1000 iterations: 1.022443
✓ Complete (2078 molecules)
  Calculating t-SNE with ECFP4... 

[11:59:34] DEPRECATION WARNING: please use MorganGenerator
[11:59:34] DEPRECATION WARNING: please use MorganGenerator
[11:59:34] DEPRECATION WARNING: please use MorganGenerator
[11:59:34] DEPRECATION WARNING: please use MorganGenerator
[11:59:34] DEPRECATION WARNING: please use MorganGenerator
[11:59:34] DEPRECATION WARNING: please use MorganGenerator
[11:59:34] DEPRECATION WARNING: please use MorganGenerator
[11:59:34] DEPRECATION WARNING: please use MorganGenerator
[11:59:34] DEPRECATION WARNING: please use MorganGenerator
[11:59:34] DEPRECATION WARNING: please use MorganGenerator
[11:59:34] DEPRECATION WARNING: please use MorganGenerator
[11:59:34] DEPRECATION WARNING: please use MorganGenerator
[11:59:34] DEPRECATION WARNING: please use MorganGenerator
[11:59:34] DEPRECATION WARNING: please use MorganGenerator
[11:59:34] DEPRECATION WARNING: please use MorganGenerator
[11:59:34] DEPRECATION WARNING: please use MorganGenerator
[11:59:34] DEPRECATION WARNING: please use MorganGenerat

[t-SNE] Computing 151 nearest neighbors...
[t-SNE] Indexed 2078 samples in 0.001s...
[t-SNE] Computed neighbors for 2078 samples in 0.120s...
[t-SNE] Computed conditional probabilities for sample 1000 / 2078
[t-SNE] Computed conditional probabilities for sample 2000 / 2078
[t-SNE] Computed conditional probabilities for sample 2078 / 2078
[t-SNE] Mean sigma: 2.398759
[t-SNE] KL divergence after 250 iterations with early exaggeration: 70.717812
[t-SNE] KL divergence after 1000 iterations: 1.048235
✓ Complete (2078 molecules)

Processing: HIGHDIV_COMBINED (2328 molecules)
  Calculating t-SNE with MACCS... [t-SNE] Computing 151 nearest neighbors...
[t-SNE] Indexed 2328 samples in 0.000s...
[t-SNE] Computed neighbors for 2328 samples in 0.035s...
[t-SNE] Computed conditional probabilities for sample 1000 / 2328
[t-SNE] Computed conditional probabilities for sample 2000 / 2328
[t-SNE] Computed conditional probabilities for sample 2328 / 2328
[t-SNE] Mean sigma: 1.807572
[t-SNE] KL divergence

[11:59:41] DEPRECATION WARNING: please use MorganGenerator
[11:59:41] DEPRECATION WARNING: please use MorganGenerator
[11:59:41] DEPRECATION WARNING: please use MorganGenerator
[11:59:41] DEPRECATION WARNING: please use MorganGenerator
[11:59:41] DEPRECATION WARNING: please use MorganGenerator
[11:59:41] DEPRECATION WARNING: please use MorganGenerator
[11:59:41] DEPRECATION WARNING: please use MorganGenerator
[11:59:41] DEPRECATION WARNING: please use MorganGenerator
[11:59:41] DEPRECATION WARNING: please use MorganGenerator
[11:59:41] DEPRECATION WARNING: please use MorganGenerator
[11:59:41] DEPRECATION WARNING: please use MorganGenerator
[11:59:41] DEPRECATION WARNING: please use MorganGenerator
[11:59:41] DEPRECATION WARNING: please use MorganGenerator
[11:59:41] DEPRECATION WARNING: please use MorganGenerator
[11:59:41] DEPRECATION WARNING: please use MorganGenerator
[11:59:41] DEPRECATION WARNING: please use MorganGenerator
[11:59:41] DEPRECATION WARNING: please use MorganGenerat

[t-SNE] Computed neighbors for 2328 samples in 0.099s...
[t-SNE] Computed conditional probabilities for sample 1000 / 2328
[t-SNE] Computed conditional probabilities for sample 2000 / 2328
[t-SNE] Computed conditional probabilities for sample 2328 / 2328
[t-SNE] Mean sigma: 2.319332
[t-SNE] KL divergence after 250 iterations with early exaggeration: 70.472748
[t-SNE] KL divergence after 1000 iterations: 0.944957
✓ Complete (2328 molecules)

Calculation complete! 4 combined t-SNEs calculated.


In [20]:
# Generate interactive HTML plots for combined datasets with legend by dataset
print("="*60)
print("GENERATING COMBINED INTERACTIVE PLOTS (HTML)")
print("="*60)

from bokeh.models import Legend, LegendItem

# Bokeh colors for each dataset
dataset_colors_bokeh = {
    'CONCAT': '#e74c3c',  # red
    'CRAFT': '#3498db',   # blue
    'LANAPDB': '#2ecc71',    # green
    'MAYBRIDGE': '#f39c12'      # orange
}

for key, data in tsne_combined_results.items():
    layout = data['layout']
    df_processed = data['df']
    diversity_type = data['diversity']
    descriptor = data['descriptor']
    perplexity = data.get('perplexity', 'N/A')
    
    print(f"Plotting: {key}...", end=' ')
    
    try:
        # Extract coordinates from t-SNE layout (n_samples, 2)
        x = layout[:, 0]
        y = layout[:, 1]
        
        # Prepare data for Bokeh
        df_plot = df_processed.copy().reset_index(drop=True)
        
        # FIX: Remove fingerprints column which is not serializable
        if 'fingerprints' in df_plot.columns:
            df_plot = df_plot.drop(columns=['fingerprints'])
            
        df_plot['x'] = x
        df_plot['y'] = y
        
        # Generate molecule images
        print("(generating images...", end=' ')
        df_plot['mol_image'] = df_plot['SMILES'].apply(mol_to_base64)
        print("OK)", end=' ')
        
        # Create figure
        p = figure(
            width=1000,
            height=800,
            title=f"Combined t-SNE - {diversity_type} - {descriptor} ({len(df_plot)} molecules unique by InChI, perplexity={perplexity})",
            tools="pan,wheel_zoom,box_zoom,reset,save",
            toolbar_location="right"
        )
        
        # List to store legend items
        legend_items = []
        
        # Add points by dataset with different colors
        for dataset_name in dataset_names:
            # Filter data for this dataset
            df_subset = df_plot[df_plot['Dataset'].str.contains(dataset_name, case=False, na=False)]
            
            if len(df_subset) > 0:
                # Create ColumnDataSource for this subset
                source = ColumnDataSource(df_subset)
                
                # Add circles
                circles = p.circle(
                    'x', 'y',
                    size=7,
                    alpha=0.7,
                    color=dataset_colors_bokeh[dataset_name],
                    source=source,
                    legend_label=f'{dataset_name} (n={len(df_subset)})'
                )
                
                # Configure HoverTool for this renderer
                hover = HoverTool(
                    tooltips="""
                    <div style="width:200px;">
                        <div>
                            <img src="@mol_image" style="width:150px; height:150px; border:1px solid #ddd; padding:5px;">
                        </div>
                        <div style="margin-top:5px;">
                            <span style="font-weight:bold;">Dataset:</span> @Dataset<br>
                            <span style="font-weight:bold;">SMILES:</span> @SMILES<br>
                            <span style="font-weight:bold;">MW:</span> @MW<br>
                            <span style="font-weight:bold;">LogP:</span> @LogP
                        </div>
                    </div>
                    """,
                    renderers=[circles]
                )
                p.add_tools(hover)
        
        # Configure axis labels
        p.xaxis.axis_label = "Dimension 1"
        p.yaxis.axis_label = "Dimension 2"
        p.title.text_font_size = "14pt"
        
        # Configure legend
        p.legend.location = "top_right"
        p.legend.click_policy = "hide"  # Allows hiding when clicking
        p.legend.background_fill_alpha = 0.8
        
        # Save as HTML
        filename = f"{diversity_type}_COMBINED_{descriptor}.html"
        filepath = os.path.join(output_dir, filename)
        output_file(filepath)
        save(p)
        
        print(f"✓ Saved")
        
    except Exception as e:
        print(f"✗ Error: {str(e)}")

print(f"\n{'='*60}")
print(f"Combined interactive plots saved in: {output_dir}")
print('='*60)

GENERATING COMBINED INTERACTIVE PLOTS (HTML)
Plotting: DRUGLIKE_COMBINED_MACCS... (generating images... OK) ✓ Saved
Plotting: DRUGLIKE_COMBINED_ECFP4... (generating images... 

OK) ✓ Saved
Plotting: HIGHDIV_COMBINED_MACCS... (generating images... 

OK) ✓ Saved
Plotting: HIGHDIV_COMBINED_ECFP4... (generating images... 

OK) ✓ Saved

Combined interactive plots saved in: /Users/francisco/Documents/Scripts/DeNovo_HsDHODH/Analysis/Results/Chemical_Spaces/TSNE


## Comparison - Druglike vs HighDiv vs Known Inhibitors

In this section we build a single t-SNE embedding combining the DRUGLIKE and HIGHDIV
generated sets together with the known HsDHODH inhibitors (Binding DB), colored by
source database. The color code matches the `physicochemical_diversity_complexity_analisys`
notebook (BuPu palette for the generated sets + pastel pink for the known inhibitors).

In [23]:
# Prepare combined data for the comparison (Druglike + HighDiv + known inhibitors)
import seaborn as sns
import matplotlib.pyplot as plt

# Load known inhibitors (same source as the physicochemical notebook)
df_inhibitors = pd.read_csv(
    '/Users/francisco/Documents/Scripts/DeNovo_HsDHODH/Analysis/Ensemble/Binding_DB/DB_curada.csv',
    sep=';'
)
df_inhibitors = df_inhibitors.rename(columns={'SMILES_curated': 'SMILES'})

# Tag each source database
df_druglike_cmp = df_druglike.copy()
df_highdiv_cmp = df_highdiv.copy()
df_druglike_cmp['Database'] = 'HsDHODH_Druglike'
df_highdiv_cmp['Database'] = 'HsDHODH_HighDiv'
df_inhibitors['Database'] = 'HsDHODH_Inhibitors'

# Consistent order and colors (same code as the physicochemical notebook)
order_desired = ['HsDHODH_Druglike', 'HsDHODH_HighDiv', 'HsDHODH_Inhibitors']
_bupu = sns.color_palette('BuPu', 4)
DB_COLORS = {
    'HsDHODH_Druglike': _bupu[1],
    'HsDHODH_HighDiv': _bupu[3],
    'HsDHODH_Inhibitors': '#F2A6C8',  # pastel pink
}

df_comparison = pd.concat(
    [df_druglike_cmp[['SMILES', 'Database']],
     df_highdiv_cmp[['SMILES', 'Database']],
     df_inhibitors[['SMILES', 'Database']]],
    ignore_index=True
)
df_comparison = df_comparison.dropna(subset=['SMILES'])

print(f"Total compounds for comparison: {len(df_comparison)}")
print(df_comparison['Database'].value_counts())

Total compounds for comparison: 5443
Database
HsDHODH_HighDiv       2328
HsDHODH_Druglike      2078
HsDHODH_Inhibitors    1037
Name: count, dtype: int64


In [24]:
# Compute the comparison t-SNE for each descriptor and plot with Bokeh (colored by database)
# Same logic/standard as the combined plots: one HTML file per descriptor.
print("="*60)
print("GENERATING COMPARISON t-SNE (Druglike vs HighDiv vs Inhibitors)")
print("="*60)

# Bokeh needs hex/CSS colors; convert the seaborn RGB tuples used above
def _to_hex(color):
    if isinstance(color, str):
        return color
    r, g, b = [int(round(c * 255)) for c in color[:3]]
    return f"#{r:02x}{g:02x}{b:02x}"

DB_COLORS_BOKEH = {db: _to_hex(c) for db, c in DB_COLORS.items()}

for descriptor in fingerprint_types:
    print(f"Plotting comparison with {descriptor}...", end=' ')

    try:
        # Create t-SNE with appropriate perplexity
        n_samples = len(df_comparison)
        perp = min(50, max(5, n_samples // 20))
        layout, df_processed = create_tsne_data(
            df_comparison.copy(), fingerprint_type=descriptor.lower(), perplexity=perp
        )

        # Extract coordinates from t-SNE layout (n_samples, 2)
        x = layout[:, 0]
        y = layout[:, 1]

        # Prepare data for Bokeh
        df_plot = df_processed.copy().reset_index(drop=True)

        # Remove fingerprints column which is not serializable
        if 'fingerprints' in df_plot.columns:
            df_plot = df_plot.drop(columns=['fingerprints'])

        df_plot['x'] = x
        df_plot['y'] = y

        # Generate molecule images
        print("(generating images...", end=' ')
        df_plot['mol_image'] = df_plot['SMILES'].apply(mol_to_base64)
        print("OK)", end=' ')

        # Create figure
        p = figure(
            width=1000,
            height=800,
            title=f"Comparison t-SNE - {descriptor} ({len(df_plot)} molecules, perplexity={perp})",
            tools="pan,wheel_zoom,box_zoom,reset,save",
            toolbar_location="right",
            output_backend="svg",  # enables vector (SVG) export
        )

        # Add points by database with different colors
        for db in order_desired:
            df_subset = df_plot[df_plot['Database'] == db]

            if len(df_subset) > 0:
                source = ColumnDataSource(df_subset)

                circles = p.scatter(
                    'x', 'y',
                    size=7,
                    alpha=0.7,
                    color=DB_COLORS_BOKEH[db],
                    source=source,
                    legend_label=f'{db} (n={len(df_subset)})'
                )

                hover = HoverTool(
                    tooltips="""
                    <div style="width:200px;">
                        <div>
                            <img src="@mol_image" style="width:150px; height:150px; border:1px solid #ddd; padding:5px;">
                        </div>
                        <div style="margin-top:5px;">
                            <span style="font-weight:bold;">Database:</span> @Database<br>
                            <span style="font-weight:bold;">SMILES:</span> @SMILES
                        </div>
                    </div>
                    """,
                    renderers=[circles]
                )
                p.add_tools(hover)

        # Configure axis labels
        p.xaxis.axis_label = "Dimension 1"
        p.yaxis.axis_label = "Dimension 2"
        p.title.text_font_size = "14pt"

        # Configure legend
        p.legend.location = "top_right"
        p.legend.click_policy = "hide"  # Allows hiding when clicking
        p.legend.background_fill_alpha = 0.8

        # Save as HTML
        filename = f"Comparison_Druglike_HighDiv_Inhibitors_{descriptor}.html"
        filepath = os.path.join(output_dir, filename)
        output_file(filepath)
        save(p)

        print(f"✓ Saved")

    except Exception as e:
        print(f"✗ Error: {str(e)}")

print(f"\n{'='*60}")
print(f"Comparison interactive plots saved in: {output_dir}")
print('='*60)

GENERATING COMPARISON t-SNE (Druglike vs HighDiv vs Inhibitors)
Plotting comparison with MACCS... [t-SNE] Computing 151 nearest neighbors...
[t-SNE] Indexed 5443 samples in 0.000s...
[t-SNE] Computed neighbors for 5443 samples in 0.152s...
[t-SNE] Computed conditional probabilities for sample 1000 / 5443
[t-SNE] Computed conditional probabilities for sample 2000 / 5443
[t-SNE] Computed conditional probabilities for sample 3000 / 5443
[t-SNE] Computed conditional probabilities for sample 4000 / 5443
[t-SNE] Computed conditional probabilities for sample 5000 / 5443
[t-SNE] Computed conditional probabilities for sample 5443 / 5443
[t-SNE] Mean sigma: 1.713971
[t-SNE] KL divergence after 250 iterations with early exaggeration: 78.424919
[t-SNE] KL divergence after 1000 iterations: 1.123163
(generating images... OK) ✓ Saved
Plotting comparison with ECFP4... 

[12:03:20] DEPRECATION WARNING: please use MorganGenerator
[12:03:20] DEPRECATION WARNING: please use MorganGenerator
[12:03:20] DEPRECATION WARNING: please use MorganGenerator
[12:03:20] DEPRECATION WARNING: please use MorganGenerator
[12:03:20] DEPRECATION WARNING: please use MorganGenerator
[12:03:20] DEPRECATION WARNING: please use MorganGenerator
[12:03:20] DEPRECATION WARNING: please use MorganGenerator
[12:03:20] DEPRECATION WARNING: please use MorganGenerator
[12:03:20] DEPRECATION WARNING: please use MorganGenerator
[12:03:20] DEPRECATION WARNING: please use MorganGenerator
[12:03:20] DEPRECATION WARNING: please use MorganGenerator
[12:03:20] DEPRECATION WARNING: please use MorganGenerator
[12:03:20] DEPRECATION WARNING: please use MorganGenerator
[12:03:20] DEPRECATION WARNING: please use MorganGenerator
[12:03:20] DEPRECATION WARNING: please use MorganGenerator
[12:03:20] DEPRECATION WARNING: please use MorganGenerator
[12:03:20] DEPRECATION WARNING: please use MorganGenerat

[t-SNE] Computing 151 nearest neighbors...
[t-SNE] Indexed 5443 samples in 0.001s...
[t-SNE] Computed neighbors for 5443 samples in 0.416s...
[t-SNE] Computed conditional probabilities for sample 1000 / 5443
[t-SNE] Computed conditional probabilities for sample 2000 / 5443
[t-SNE] Computed conditional probabilities for sample 3000 / 5443
[t-SNE] Computed conditional probabilities for sample 4000 / 5443
[t-SNE] Computed conditional probabilities for sample 5000 / 5443
[t-SNE] Computed conditional probabilities for sample 5443 / 5443
[t-SNE] Mean sigma: 2.369663
[t-SNE] KL divergence after 250 iterations with early exaggeration: 79.055214
[t-SNE] KL divergence after 1000 iterations: 1.165944
(generating images... OK) ✓ Saved

Comparison interactive plots saved in: /Users/francisco/Documents/Scripts/DeNovo_HsDHODH/Analysis/Results/Chemical_Spaces/TSNE
